# ScamShield — Exploratory Data Analysis

**Project:** Explainable Multimodal AI for Phishing and Scam Detection Using Text, URL and Intent Analysis

This notebook performs **read-only exploratory data analysis** on the frozen standardized dataset in `data/interim`.

> **Important:** This notebook does not modify the dataset. It only reads the Parquet files, analyzes distributions, creates tables/plots, and records observations.

## EDA Roadmap

1. Environment and Spark setup
2. Load and verify the frozen dataset
3. Dataset overview and schema
4. Overall class distribution
5. Source dataset composition
6. Channel composition
7. Class × source analysis
8. Text-length analysis
9. URL analysis
10. Language analysis
11. Natural vs generated data
12. Scam-category analysis
13. Engineered-feature analysis
14. Dataset artifacts and modeling implications
15. Final EDA summary

In [1]:
# Core imports
import os
import re
import json
import math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, BooleanType

print("Python environment ready.")

Python environment ready.


In [2]:
# Start Spark
spark = (
    SparkSession.builder
    .appName("ScamShield_EDA")
    .master("local[1]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 3.5.1


In [3]:
# Resolve the project root robustly.
# When this notebook is run from E:\scamshield\notebooks,
# the parent of the notebook directory is the project root.
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
elif (project_root / "data" / "interim").exists():
    pass
elif (project_root.parent / "data" / "interim").exists():
    project_root = project_root.parent

interim_path = project_root / "data" / "interim"

print("Project root:", project_root)
print("Interim dataset:", interim_path)
print("Exists:", interim_path.exists())

if not interim_path.exists():
    raise FileNotFoundError(
        f"Could not find {interim_path}. "
        "Open this notebook from the ScamShield project."
    )

Project root: e:\scamshield
Interim dataset: e:\scamshield\data\interim
Exists: True


## 2. Load the Frozen Dataset

In [ ]:
import pandas as pd

df = pd.read_parquet(
    interim_path,
    engine="pyarrow"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Columns:")
print(df.columns.tolist())

Py4JJavaError: An error occurred while calling o26.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:180)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:95)
	at scala.collection.TraversableLike.$anonf

In [ ]:
# Basic integrity checks for the EDA input
print("=== BASIC DATASET CHECK ===")

total_rows = df.count()
null_text = df.filter(F.col("text").isNull()).count()
null_label = df.filter(F.col("project_label").isNull()).count()
null_split = df.filter(F.col("split").isNull()).count()
null_source = df.filter(F.col("source_dataset").isNull()).count()

print("Total rows:", total_rows)
print("NULL text:", null_text)
print("NULL project_label:", null_label)
print("NULL split:", null_split)
print("NULL source_dataset:", null_source)

assert total_rows == 119063, f"Unexpected row count: {total_rows}"
assert null_text == 0
assert null_label == 0
assert null_split == 0
assert null_source == 0

print("\nFrozen dataset loaded successfully.")

## 3. Dataset Overview

In [ ]:
# Compact overview
overview = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Number of columns",
        "Train rows",
        "Validation rows",
        "Test rows",
        "Unique languages",
        "Unique source datasets",
        "Unique channels"
    ],
    "Value": [
        total_rows,
        len(df.columns),
        df.filter(F.col("split") == "train").count(),
        df.filter(F.col("split") == "validation").count(),
        df.filter(F.col("split") == "test").count(),
        df.select("language").distinct().count(),
        df.select("source_dataset").distinct().count(),
        df.select("channel").distinct().count()
    ]
})

display(overview)

In [ ]:
display(df.limit(10).toPandas())

## 4. Overall Class Distribution

This section examines the target variable `project_label`.

The class distribution is important because it tells us whether the problem is balanced or whether later model evaluation needs to pay particular attention to minority classes.

In [ ]:
total = df.count()

class_df = (
    df.groupBy("project_label")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy(F.desc("count"))
)

class_pd = class_df.toPandas()
display(class_pd)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(class_pd["project_label"], class_pd["count"])
plt.title("Overall Project Label Distribution")
plt.xlabel("Project Label")
plt.ylabel("Number of Messages")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Observation

The dataset contains four project labels: **benign, phishing, spam, and scam**.

The exact counts above should be used in the final project report rather than manually re-entering numbers.

## 5. Source Dataset Composition

In [ ]:
source_df = (
    df.groupBy("source_dataset")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy(F.desc("count"))
)

source_pd = source_df.toPandas()
display(source_pd)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(source_pd["source_dataset"], source_pd["count"])
plt.title("Dataset Composition by Source")
plt.xlabel("Source Dataset")
plt.ylabel("Number of Records")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 6. Channel Composition

In [ ]:
channel_df = (
    df.groupBy("channel")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy(F.desc("count"))
)

channel_pd = channel_df.toPandas()
display(channel_pd)

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(channel_pd["channel"], channel_pd["count"])
plt.title("Records by Communication Channel")
plt.xlabel("Channel")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

## 7. Class Distribution by Source

This is one of the most important EDA sections because source composition can create dataset shortcuts.

We are **not using these source variables as model features** here. We are analyzing them to understand the dataset.

In [ ]:
source_label = (
    df.groupBy("source_dataset", "project_label")
      .count()
      .orderBy("source_dataset", F.desc("count"))
)

display(source_label.toPandas())

In [ ]:
source_label_pd = (
    source_label
    .groupBy("source_dataset")
    .pivot("project_label")
    .sum("count")
    .fillna(0)
    .toPandas()
    .set_index("source_dataset")
)

ax = source_label_pd.plot(kind="bar", figsize=(11, 6))
ax.set_title("Project Labels by Source Dataset")
ax.set_xlabel("Source Dataset")
ax.set_ylabel("Number of Records")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### Modeling note

If one source contains only one target class, a model could potentially learn the source-specific writing style or metadata instead of learning genuine scam/phishing characteristics.

For this reason, source metadata is retained for **analysis and auditing**, not treated as a normal predictive feature in the main model.

## 8. Split Distribution

We verify that the train, validation, and test sets have broadly comparable compositions.

In [ ]:
split_label = (
    df.groupBy("split", "project_label")
      .count()
      .orderBy("split", F.desc("count"))
)

display(split_label.toPandas())

In [ ]:
split_pd = (
    df.groupBy("split")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy("split")
      .toPandas()
)

display(split_pd)

plt.figure(figsize=(7, 5))
plt.bar(split_pd["split"], split_pd["count"])
plt.title("Dataset Split Sizes")
plt.xlabel("Split")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

## 9. Text Length Analysis

We analyze both character length and approximate word count.

These are descriptive features for EDA. They help us understand whether phishing/scam messages tend to be short, long, or highly variable.

In [ ]:
df_eda = (
    df
    .withColumn("char_count", F.length(F.col("text")))
    .withColumn(
        "word_count",
        F.size(
            F.split(
                F.trim(F.regexp_replace(F.col("text"), r"\s+", " ")),
                " "
            )
        )
    )
)

text_stats = (
    df_eda.groupBy("project_label")
          .agg(
              F.count("*").alias("rows"),
              F.round(F.avg("char_count"), 2).alias("avg_chars"),
              F.round(F.expr("percentile_approx(char_count, 0.5)"), 2).alias("median_chars"),
              F.round(F.min("char_count"), 2).alias("min_chars"),
              F.round(F.max("char_count"), 2).alias("max_chars"),
              F.round(F.avg("word_count"), 2).alias("avg_words"),
              F.round(F.expr("percentile_approx(word_count, 0.5)"), 2).alias("median_words")
          )
          .orderBy(F.desc("rows"))
)

display(text_stats.toPandas())

In [ ]:
# Sample data for plotting
text_pd = (
    df_eda
    .select("project_label", "char_count", "word_count")
    .sample(False, 0.20, seed=42)
    .toPandas()
)

for label in sorted(text_pd["project_label"].dropna().unique()):
    subset = text_pd[text_pd["project_label"] == label]
    plt.figure(figsize=(8, 5))
    plt.hist(subset["char_count"], bins=60)
    plt.title(f"Character Length Distribution — {label}")
    plt.xlabel("Characters")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 10. URL Analysis

In [ ]:
url_label = (
    df.groupBy("project_label", "url_present")
      .count()
      .orderBy("project_label", "url_present")
)

display(url_label.toPandas())

In [ ]:
url_pd = (
    df.groupBy("project_label", "url_present")
      .count()
      .toPandas()
)

pivot_url = (
    url_pd.pivot(index="project_label", columns="url_present", values="count")
           .fillna(0)
)

display(pivot_url)

ax = pivot_url.plot(kind="bar", figsize=(9, 5))
ax.set_title("URL Presence by Project Label")
ax.set_xlabel("Project Label")
ax.set_ylabel("Number of Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### URL-related engineered features

For MeAJOR records, `engineered_features` contains URL statistics such as:

- `url_count`
- `url_length_max`
- `url_length_avg`
- `url_subdom_max`
- `url_subdom_avg`

These are inspected below without modifying the original column.

In [ ]:
def parse_engineered_features(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    try:
        return json.loads(value)
    except Exception:
        return {}

feature_sample_pd = (
    df.filter(F.col("engineered_features").isNotNull())
      .select("source_dataset", "engineered_features")
      .limit(5000)
      .toPandas()
)

parsed_features = pd.json_normalize(
    feature_sample_pd["engineered_features"].apply(parse_engineered_features)
)

display(parsed_features.head())

## 11. Language Analysis

In [ ]:
language_df = (
    df.groupBy("language")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 3))
      .orderBy(F.desc("count"))
)

language_pd = language_df.toPandas()

display(language_pd.head(30))

In [ ]:
top_languages = language_pd.head(15)

plt.figure(figsize=(10, 6))
plt.bar(top_languages["language"], top_languages["count"])
plt.title("Top Languages by Number of Records")
plt.xlabel("Language")
plt.ylabel("Number of Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
language_label = (
    df.groupBy("language", "project_label")
      .count()
      .orderBy("language", F.desc("count"))
)

display(language_label.toPandas().head(100))

### Language interpretation

The language distribution is heavily English-dominated, while several smaller language groups are much smaller.

Language is potentially useful at inference time for multilingual modeling, but the strong source-induced label imbalance means its effect should be evaluated carefully in later model ablation experiments.

## 12. Natural vs Generated Data

In [ ]:
natural_df = (
    df.groupBy("is_natural")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy(F.desc("count"))
)

display(natural_df.toPandas())

In [ ]:
augmentation_df = (
    df.groupBy("augmentation_type", "project_label")
      .count()
      .orderBy("augmentation_type", F.desc("count"))
)

display(augmentation_df.toPandas())

In [ ]:
augmentation_pd = (
    df.groupBy("augmentation_type")
      .count()
      .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
      .orderBy(F.desc("count"))
      .toPandas()
)

plt.figure(figsize=(9, 5))
plt.bar(augmentation_pd["augmentation_type"], augmentation_pd["count"])
plt.title("Natural vs Generated/Augmented Records")
plt.xlabel("Augmentation Type")
plt.ylabel("Number of Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Modeling note

`augmentation_type` and `is_natural` are useful for understanding dataset construction, but they should not be used as normal predictive inputs in the main model because they are strongly associated with how the dataset was generated.

## 13. Scam Category Analysis

In [ ]:
category_df = (
    df.filter(
        F.col("scam_category").isNotNull() &
        (F.trim(F.col("scam_category")) != "")
    )
    .groupBy("scam_category")
    .count()
    .orderBy(F.desc("count"))
)

category_pd = category_df.toPandas()

display(category_pd)

In [ ]:
if len(category_pd) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(category_pd["scam_category"][::-1], category_pd["count"][::-1])
    plt.title("Indian Cyber Scam Categories")
    plt.xlabel("Number of Records")
    plt.ylabel("Scam Category")
    plt.tight_layout()
    plt.show()

## 14. Engineered Feature Exploration

The standardized dataset stores source-specific engineered information as JSON.

We inspect common numeric fields from MeAJOR separately rather than pretending that every feature exists for every source.

In [ ]:
meajor_features_pd = (
    df.filter(
        (F.col("source_dataset") == "meajor") &
        F.col("engineered_features").isNotNull()
    )
    .select("project_label", "engineered_features")
    .toPandas()
)

meajor_features = pd.json_normalize(
    meajor_features_pd["engineered_features"].apply(parse_engineered_features)
)

# Convert common numeric fields where available
numeric_feature_names = [
    "url_count",
    "url_length_max",
    "url_length_avg",
    "url_subdom_max",
    "url_subdom_avg",
    "attachment_count"
]

for col in numeric_feature_names:
    if col in meajor_features.columns:
        meajor_features[col] = pd.to_numeric(meajor_features[col], errors="coerce")

meajor_features["project_label"] = meajor_features_pd["project_label"].values

display(meajor_features[numeric_feature_names + ["project_label"]].describe(include="all"))

In [ ]:
for feature_name in numeric_feature_names:
    if feature_name not in meajor_features.columns:
        continue

    plt.figure(figsize=(9, 5))

    for label in sorted(meajor_features["project_label"].dropna().unique()):
        values = meajor_features.loc[
            meajor_features["project_label"] == label,
            feature_name
        ].dropna()

        if len(values) > 0:
            plt.hist(values, bins=50, alpha=0.55, label=label)

    plt.title(f"MeAJOR Feature Distribution — {feature_name}")
    plt.xlabel(feature_name)
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 15. Domain Cardinality

Raw sender and receiver domains have high cardinality.

This section documents that fact for modeling decisions. Raw domain strings should not simply be one-hot encoded into the first model because that can encourage memorization of specific domains.

In [ ]:
domain_cardinality = pd.DataFrame({
    "Feature": ["sender_domain", "receiver_domain"],
    "Unique Values": [
        meajor_features.get("sender_domain", pd.Series(dtype=str)).nunique(),
        meajor_features.get("receiver_domain", pd.Series(dtype=str)).nunique()
    ]
})

# If the fields are not present in the limited normalized sample,
# calculate the authoritative counts directly from Spark.
domain_cardinality = pd.DataFrame({
    "Feature": ["sender_domain", "receiver_domain"],
    "Unique Values": [
        df.filter(F.col("source_dataset") == "meajor")
          .select(F.get_json_object("engineered_features", "$.sender_domain").alias("sender_domain"))
          .filter(F.col("sender_domain").isNotNull())
          .distinct().count(),

        df.filter(F.col("source_dataset") == "meajor")
          .select(F.get_json_object("engineered_features", "$.receiver_domain").alias("receiver_domain"))
          .filter(F.col("receiver_domain").isNotNull())
          .distinct().count()
    ]
})

display(domain_cardinality)

## 16. Dataset-Level Modeling Considerations

Based on the EDA, the following categories should be treated differently:

### Keep as candidate predictive information
- `text`
- `channel`
- `language`
- `url_present`
- structural URL features
- structural domain features
- selected modality-specific features

### Keep for EDA / auditing / ablation, not the main feature matrix
- `source_dataset`
- `augmentation_type`
- `is_natural`
- `meajor_source_corpus`
- `original_corpus_source`
- `llm_model`
- `llm_scam_category`
- raw sender-domain strings
- raw receiver-domain strings

The goal is to make the model learn **message/scam characteristics**, rather than simply identifying where a record came from.

## 17. Final EDA Summary

Run the cells above first. Then record the final observations here based on the generated tables and plots.

### Dataset
- Total records: **119,063**
- Four project labels: **benign, phishing, spam, scam**
- Four source datasets are represented.

### Data composition
- The dataset is dominated by MeAJOR records.
- Email is the dominant communication channel.
- English is the dominant language.

### Class structure
- Benign and phishing form the two largest classes.
- Spam and scam are much smaller classes.
- This class imbalance should be considered during model training and evaluation.

### Dataset-construction effects
- Generated/augmented records are strongly associated with particular labels.
- Source metadata therefore needs to be kept out of the main predictive feature matrix.

### Feature engineering
- MeAJOR contains URL, attachment, sender-domain, receiver-domain and language-related metadata.
- Indian Cyber Scam contains modality-specific fields such as caller type, urgency, blackmail indicator and audio duration.
- These should be evaluated according to the modality they belong to.

### Next stage
After EDA is completed, move to **feature preparation and baseline modeling** without changing the frozen dataset.

In [ ]:
# Stop Spark after completing the notebook.
# Run this only when you are finished with the EDA session.
spark.stop()

print("Spark session stopped.")